In [3]:
import os
import pandas as pd
from datasets import load_dataset

import urllib3

# Убираем сетевые проверки SSL для безопасности
os.environ["CURL_CA_BUNDLE"] = ""
os.environ["PYTHONHTTPSVERIFY"] = "0"

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [ ]:
# Загружаем датасет (он мгновенно подтянется из локального кэша)
dataset = load_dataset('IlyaGusev/gazeta')

# Переводим тренировочную выборку в DataFrame для удобного анализа
df = pd.DataFrame(dataset['train'])

# Выведем названия всех колонок и первые 5 строк
print("Доступные колонки:", df.columns.tolist())
df.head()

Доступные колонки: ['text', 'summary', 'title', 'date', 'url']


,text,summary,title,date,url
0,Сегодня транспортный налог начисляется в завис...,С 2011 года правительство отменяет самый раздр...,Налог в бак,2010-06-01 10:35:49,https://www.gazeta.ru/auto/2010/05/31_a_337771...
1,Словосочетание «музыкальный кинофестиваль» уже...,"Британские затворники, московские модники, бра...","Секс, наркотики и темный зал",2010-06-01 10:42:59,https://www.gazeta.ru/culture/2010/06/01/a_337...
2,После более чем 12-часовых консультаций Совет ...,Совбез ООН собрался на экстренное совещание дл...,Осудить и отпустить,2010-06-01 11:00:30,https://www.gazeta.ru/politics/2010/06/01_a_33...
3,"Жертвами урагана «Агата», обрушившегося на Цен...",Ураган «Агата» в Центральной Америке унес жизн...,«Агата» открыла страшный сезон,2010-06-01 11:05:30,https://www.gazeta.ru/social/2010/06/01/337799...
4,Решение ограничить рост тарифов естественных м...,Правительство хочет сдержать рост тарифов есте...,Тарифы инфляцию не остановят,2010-06-01 11:48:50,https://www.gazeta.ru/financial/2010/06/01/337...


1. URL содержат тематику новости: culture, politics и другие
2. Темы для классификатора возьмем из URL

In [7]:
# Функция для извлечения темы из URL
# Ссылка выглядит так: https://gazeta.ru
# Мы берем часть после "gazeta.ru/" (это индекс 3 после разделения по "/")
def extract_topic(url):
    parts = url.split('/')
    if len(parts) > 3:
        return parts[3]
    return 'unknown'

# Создаем новую колонку с темами
df['topic'] = df['url'].apply(extract_topic)

# Выводим список тем и количество новостей в каждой из них
print("--- Распределение новостей по темам в датасете ---")
print(df['topic'].value_counts().head(10))

--- Распределение новостей по темам в датасете ---
topic
sport        12654
social       10066
politics      9564
business      7689
culture       7416
army          3401
auto          2995
science       1553
financial     1504
tech          1337
Name: count, dtype: int64


In [5]:
# Загружаем DataLoader'ы (датасет уже должен быть загружен локально)
from src.dataset import get_data_loaders

train_loader, val_loader = get_data_loaders()

# Заводим словарь для сохранения меток классов
results_archive = {}

Загружаем отфильтрованную базу данных...
Словарь тем зафиксирован: {'army': 0, 'auto': 1, 'financial': 2, 'science': 3, 'tech': 4}
✅ Датасет успешно сэмплирован! Выборка составляет 7904 строк.
Инициализируем токенизатор Сбера...
Данные готовы. Батчей в Train: 396, в Val: 99


In [2]:
import torch
import torch.nn as nn
from torch.optim import AdamW

from src.models.pooling import MeanPooling
from src.models.classifier import BertClassifier
from src.trainer import train_epoch, evaluate
from src.config import Config

c:\Users\pelse\Documents\education\МАГИСТРАТУРА_ИТМО\AI_conversational\AI_lab_1_classifier\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Конфигурация загружена. Выбранное устройство: cpu


## Эксперимент №1 — Mean Pooling

In [4]:
print(f"--- Старт Эксперимента 1: Mean Pooling на {Config.DEVICE} ---")

# Инициализируем модель с MeanPooling
model_mean = BertClassifier(pooling_layer=MeanPooling(), num_classes=7).to(Config.DEVICE)
optimizer = AdamW(model_mean.parameters(), lr=Config.LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

# Обучаем (для теста ЛР можно поставить 1 эпоху, чтобы проверить скорость, или Config.EPOCHS)
for epoch in range(1): 
    train_loss = train_epoch(model_mean, train_loader, optimizer, criterion)
    val_metrics = evaluate(model_mean, val_loader, criterion)
    print(f"Эпоха {epoch+1} завершена. Валидация: {val_metrics}")

# Сохраняем метрики в архив
results_archive["Mean Pooling"] = {
    "Precision": val_metrics["Precision"],
    "Recall": val_metrics["Recall"],
    "F1-score": val_metrics["F1-score"],
    "Accuracy": val_metrics["Accuracy"]
}

--- Старт Эксперимента 1: Mean Pooling на cpu ---


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11817.45it/s]
[transformers] BertModel LOAD REPORT from: ai-forever/ruBert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Evaluating: 100%|██████████| 99/99 [03:44<00:00,  2.26s/it]

Эпоха 1 завершена. Валидация: {'Accuracy': 0.9589, 'Precision': 0.9591, 'Recall': 0.9589, 'F1-score': 0.9589, 'Loss': 0.1224}


In [11]:
import json

# Впишите сюда вручную ваши метрики из лога валидации Mean Pooling, если они вывелись на экран:
mean_metrics = {
    "Precision": 0.9591,  # Замените 0.0 на ваше число из лога (например, 0.82)
    "Recall": 0.9589,     # Замените 0.0 на ваше число из лога
    "F1-score": 0.9589,   # Замените 0.0 на ваше число из лога
    "Accuracy": 0.9589,
    "Loss": 0.1224  # Замените 0.0 на ваше число из лога
}

# Сохраняем в файл на диске
with open("mean_pooling_metrics.json", "w") as f:
    json.dump(mean_metrics, f)

print("Метрики Mean Pooling успешно спасены на диск в файл mean_pooling_metrics.json!")

Метрики Mean Pooling успешно спасены на диск в файл mean_pooling_metrics.json!


## Эксперимент №2 — CLS Pooling

In [7]:
from src.models.pooling import ClsPooling

print(f"\n--- Старт Эксперимента 2: CLS Pooling на {Config.DEVICE} ---")

# Инициализируем точно такую же модель, но с ClsPooling
model_cls = BertClassifier(pooling_layer=ClsPooling(), num_classes=5).to(Config.DEVICE)
optimizer = AdamW(model_cls.parameters(), lr=Config.LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

for epoch in range(1):
    train_loss = train_epoch(model_cls, train_loader, optimizer, criterion)
    val_metrics = evaluate(model_cls, val_loader, criterion)
    print(f"Эпоха {epoch+1} завершена. Валидация: {val_metrics}")

results_archive["CLS Pooling"] = {
    "Precision": val_metrics["Precision"],
    "Recall": val_metrics["Recall"],
    "F1-score": val_metrics["F1-score"],
    "Accuracy": val_metrics["Accuracy"]
}


--- Старт Эксперимента 2: CLS Pooling на cpu ---


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15317.51it/s]
[transformers] BertModel LOAD REPORT from: ai-forever/ruBert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Evaluating: 100%|██████████| 99/99 [03:52<00:00,  2.35s/it]

Эпоха 1 завершена. Валидация: {'Accuracy': 0.9671, 'Precision': 0.9673, 'Recall': 0.9671, 'F1-score': 0.9671, 'Loss': 0.1195}


In [9]:
import json

# Впишите сюда вручную ваши метрики из лога валидации Mean Pooling, если они вывелись на экран:
mean_metrics = {
    "Precision": 0.9673,  # Замените 0.0 на ваше число из лога (например, 0.82)
    "Recall": 0.9671,     # Замените 0.0 на ваше число из лога
    "F1-score": 0.9671,   # Замените 0.0 на ваше число из лога
    "Accuracy": 0.9671,
    "Loss": 0.1195  # Замените 0.0 на ваше число из лога
}

# Сохраняем в файл на диске
with open("cls_pooling_metrics.json", "w") as f:
    json.dump(mean_metrics, f)

print("Метрики CLS Pooling успешно спасены на диск в файл cls_pooling_metrics.json!")

Метрики CLS Pooling успешно спасены на диск в файл cls_pooling_metrics.json!


## Сравнение метрик

In [17]:
import json
import pandas as pd

# 1. Читаем метрики из сохраненных JSON-файлов
with open("mean_pooling_metrics.json", "r", encoding="utf-8") as f:
    mean_data = json.load(f)

with open("cls_pooling_metrics.json", "r", encoding="utf-8") as f:
    cls_data = json.load(f)

# 2. Собираем данные в общий словарь для таблицы
comparison_dict = {
    "Mean Pooling": {
        "Accuracy": mean_data["Accuracy"],
        "Precision": mean_data["Precision"],
        "Recall": mean_data["Recall"],
        "F1-score": mean_data["F1-score"],
        "Loss": mean_data["Loss"]
    },
    "CLS Pooling": {
        "Accuracy": cls_data["Accuracy"],
        "Precision": cls_data["Precision"],
        "Recall": cls_data["Recall"],
        "F1-score": cls_data["F1-score"],
        "Loss": cls_data["Loss"]
    }
}

# 3. Переводим в DataFrame Pandas и транспонируем (.T)
df_compare = pd.DataFrame(comparison_dict).T

print("================== СРАВНИТЕЛЬНЫЙ АНАЛИЗ СТРАТЕГИЙ ПУЛИНГА ==================")

# Применяем контрастную темную подсветку с принудительным белым цветом текста
styled_table = df_compare.style\
    .highlight_max(
        subset=["Accuracy", "Precision", "Recall", "F1-score"], 
        axis=0, 
        props="background-color: #1b5e20; color: white;"
    )\
    .highlight_min(
        subset=["Loss"], 
        axis=0, 
        props="background-color: #0d47a1; color: white;"
    )

display(styled_table)


================== СРАВНИТЕЛЬНЫЙ АНАЛИЗ СТРАТЕГИЙ ПУЛИНГА ==================


,Accuracy,Precision,Recall,F1-score,Loss
Mean Pooling,0.958900,0.959100,0.958900,0.958900,0.122400
CLS Pooling,0.967100,0.967300,0.967100,0.967100,0.119500
